In [1]:
import numpy as np
from mpi4py import MPI

import dolfinx
import dolfinx.plot
import ufl
from ufl import grad, inner, dot, jump, avg, FacetNormal, CellDiameter, Measure, Mesh, MixedFunctionSpace, TrialFunctions, TestFunctions, pi, SpatialCoordinate
import basix.ufl

from fenicsx_ii import LinearProblem, Average, Circle

from mesh import GenMesh3D


In [7]:
N = 4
comm = MPI.COMM_WORLD

nodes3D, cells3D, graph_nodes, graph_cells = GenMesh3D(N)

ufl_tri = Mesh(basix.ufl.element("Lagrange", "prism", 1, shape=(3,)))
ufl_int = Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(3,)))

mesh_3d = dolfinx.mesh.create_mesh(
    MPI.COMM_WORLD, cells=cells3D, x=nodes3D, e=ufl_tri
)

mesh_1d = dolfinx.mesh.create_mesh(
    MPI.COMM_WORLD, cells=graph_cells, x=graph_nodes, e=ufl_int
)

[52. 57. 62. 66. 70. 68. 74.]


In [8]:
# Define Function Spaces

degree=1
V = dolfinx.fem.functionspace(mesh_3d, ("CG", degree))
Q = dolfinx.fem.functionspace(mesh_1d, ("CG", degree))

W = MixedFunctionSpace(*[V, Q])
(u, uhat) = TrialFunctions(W)
(w, what) = TestFunctions(W)

# Define Restriction Functions

radius = 0.05
q_degree = 20
restriction_trial = Circle(mesh_1d, radius, degree=q_degree)
restriction_test = Circle(mesh_1d, radius, degree=q_degree)

q_el = basix.ufl.quadrature_element(mesh_1d.basix_cell(), value_shape=(), degree=q_degree)
restriction_space = dolfinx.fem.functionspace(mesh_1d, q_el)

avg_u = Average(u, restriction_trial, restriction_space)
avg_w = Average(w, restriction_test, restriction_space)

# Define integraiton measures

dx_3D = Measure("dx", domain=mesh_3d)
dS_3D = Measure("dS", domain=mesh_3d)
ds_3D = Measure("ds", domain=mesh_3d)
dx_1D = Measure("dx", domain=mesh_1d)
dS_1D = Measure("dS", domain=mesh_1d)
ds_1D = Measure("ds", domain=mesh_1d)

In [ ]:
# Define Bilinear and Linear Forms

# Cell normals/sizes

n_2D = FacetNormal(mesh_3d)
h_2D = CellDiameter(mesh_3d)
h_2D_avg = (h_2D("+") + h_2D("-"))/2
n_1D = FacetNormal(mesh_1d)
h_1D = CellDiameter(mesh_1d)
h_1D_avg = (h_1D("+") + h_1D("-"))/2

# physical parameters

alpha1 = dolfinx.fem.Constant(mesh_3d, 1.0)
area = ufl.pi * radius ** 2
perimeter = 2 * ufl.pi * radius

# penalty parameters

epsilon_1 = -1
sigma_2D = 30
epsilon_2 = -1
sigma_1D = 30

# spatially dependent constants

xi = dolfinx.fem.Constant(mesh_3d, 1.0)

# Define the bilinear form

a = inner(grad(u), grad(w)) * dx_3D

# 1D component

a += inner(grad(uhat), grad(what)) * dx_1D

# coupling components

a += perimeter * xi * inner(avg_u - uhat, avg_w - what) * dx_1D

# Define exact solutions

x3D = SpatialCoordinate(mesh_3d)
x1D = SpatialCoordinate(mesh_1d)

# might need to use 3d coordinate

uhat_1 = x1D[2] + ufl.cos(2 * pi * x1D[2])
uhat_2 = 2 + 0.5 * ufl.sqrt(2) * (x1D[2] - 1)
uhat_ex = ufl.conditional(x1D[2] < 1, uhat_1, uhat_2)

xi_rate = xi / (xi + 1)

r = ufl.sqrt(x3D[0] ** 2 + x3D[1]**2)
u_inside = xi_rate * uhat_ex
u_outside_1 = xi_rate * uhat_ex * (1 - radius * ufl.ln(r / radius))
u_1 = ufl.conditional(r <= radius, u_inside, u_outside_1)

u_outside_2 = ufl.conditional

# RHS function

bc_expr = dolfinx.fem.Expression(u_ex, V.element.interpolation_points)
u_bc = dolfinx.fem.Function(V)
u_bc.interpolate(bc_expr)

bc_hat_expr = dolfinx.fem.Expression(uhat_ex, Q.element.interpolation_points)
uhat_bc = dolfinx.fem.Function(Q)
uhat_bc.interpolate(bc_hat_expr)

f_vol = -ufl.div(ufl.grad(u_ex))
A_fhat  = - ufl.div(area * ufl.grad(uhat_ex)) + perimeter * xi_rate * uhat_ex

L = ufl.inner(f_vol, w) * dx_2D
L += ufl.inner(A_fhat, what) * dx_1D


NameError: name 'x2D' is not defined